## import Data

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import GRU 
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from pathlib import Path
import joblib

In [18]:
def get_all_files(directory, max_files=None):
    return [str(f) for f in Path(directory).rglob("*.parquet") if not f.name.endswith("adsc.parquet")][:max_files]

In [19]:
input_folder = "../data/processed"
file_list = get_all_files(input_folder,3500)
len(file_list)

3500

In [20]:
df_list = [pd.read_parquet(file) for file in file_list]


In [21]:
len(df_list)

3500

## Train-Test split

In [ ]:
FEATURE_COLS = ["latitude", "longitude", "altitude",
                "vx", "vy", "acceleration",
                "turn_rate"]
TARGET_COLS  = ["latitude", "longitude", "altitude"]
SEQUENCE_LEN = 10

In [23]:
def great_circle_extrapolate(lat1, lon1, lat2, lon2):
    """Extrapolate one step forward from two points using great-circle projection."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Bearing from point 1 to point 2
    dlon = lon2 - lon1
    x = np.cos(lat2) * np.sin(dlon)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = np.arctan2(x, y)
    
    # Angular distance between point 1 and point 2
    dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    angular_dist = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    # Project from point 2 with same bearing and distance
    lat3 = np.arcsin(np.sin(lat2) * np.cos(angular_dist) +
                     np.cos(lat2) * np.sin(angular_dist) * np.cos(bearing))
    lon3 = lon2 + np.arctan2(np.sin(bearing) * np.sin(angular_dist) * np.cos(lat2),
                              np.cos(angular_dist) - np.sin(lat2) * np.sin(lat3))
    
    return np.degrees(lat3), np.degrees(lon3)


def create_windows_relative(flight_df, sequence_len, feature_cols, target_cols):
    X, y, gc = [], [], []

    flight_df = flight_df.sort_values("timestamp").reset_index(drop=True)
    flight_df = flight_df.dropna()

    if len(flight_df) < sequence_len + 1:
        return np.array(X), np.array(y), np.array(gc)

    features = flight_df[feature_cols].values
    targets  = flight_df[target_cols].values
    
    # Raw lat/lon/alt for great-circle computation
    raw_lat = flight_df["latitude"].values
    raw_lon = flight_df["longitude"].values
    raw_alt = flight_df["altitude"].values

    for i in range(len(flight_df) - sequence_len):
        window = features[i : i + sequence_len].copy()
        target = targets[i + sequence_len].copy()

        # Store the last point as reference
        ref_lat = window[-1, 0]
        ref_lon = window[-1, 1]
        ref_alt = window[-1, 2]

        # Make lat/lon/alt relative to last point in window
        window[:, 0] -= ref_lat
        window[:, 1] -= ref_lon
        window[:, 2] -= ref_alt

        target[0] -= ref_lat
        target[1] -= ref_lon
        target[2] -= ref_alt

        # Great-circle extrapolation from last two raw points
        gc_lat, gc_lon = great_circle_extrapolate(
            raw_lat[i + sequence_len - 2], raw_lon[i + sequence_len - 2],
            raw_lat[i + sequence_len - 1], raw_lon[i + sequence_len - 1]
        )
        gc_alt = raw_alt[i + sequence_len - 1]

        # gc_guess also relative to the same reference point
        gc_guess = [gc_lat - ref_lat, gc_lon - ref_lon, gc_alt - ref_alt]

        X.append(window)
        y.append(target)
        gc.append(gc_guess)

 

    return np.array(X), np.array(y), np.array(gc)

In [24]:
def split_at_outliers(df, max_dlat=0.05, max_dlon=0.05, max_dalt=500):
    """Split a flight into segments at physically impossible jumps."""
    df = df.copy().sort_values("timestamp").reset_index(drop=True)
    
    dlat = df["latitude"].diff().abs()
    dlon = df["longitude"].diff().abs()
    dalt = df["altitude"].diff().abs()
    
    bad = (dlat > max_dlat) | (dlon > max_dlon) | (dalt > max_dalt)
    split_indices = bad[bad].index.tolist()
    
    segments = []
    prev = 0
    for idx in split_indices:
        if idx - prev >= 2:
            segments.append(df.iloc[prev:idx].copy())
        prev = idx
    if len(df) - prev >= 2:
        segments.append(df.iloc[prev:].copy())
    
    return segments

In [25]:
from sklearn.model_selection import train_test_split

from pathlib import Path
import numpy as np
import pandas as pd
# ── Load and window all flights ────────────────────────────────────────────────

all_X, all_y, all_gc = [], [], []

for flight_df in df_list:
    segments = split_at_outliers(flight_df, max_dlat=0.1, max_dlon=0.1, max_dalt=1000)
    for seg in segments:
        seg = seg.loc[~(seg[["latitude", "longitude", "altitude"]].diff() == 0).all(axis=1)].reset_index(drop=True)
        
        X, y, gc = create_windows_relative(seg, SEQUENCE_LEN, FEATURE_COLS, TARGET_COLS)
        if len(X) == 0:
            continue
        all_X.append(X)
        all_y.append(y)
        all_gc.append(gc)

all_X = np.concatenate(all_X, axis=0)
all_y = np.concatenate(all_y, axis=0)
all_gc = np.concatenate(all_gc, axis=0)

print(f"Total windows: {all_X.shape[0]}")
print(f"X shape: {all_X.shape}")
print(f"y shape: {all_y.shape}")
print(f"gc shape: {all_gc.shape}")

# ── Split ─────────────────────────────────────────────────────────────────────
indices = np.arange(len(all_X))
train_idx, test_idx = train_test_split(indices, test_size=0.15, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.15, random_state=42)

X_train, X_val, X_test = all_X[train_idx], all_X[val_idx], all_X[test_idx]
y_train, y_val, y_test = all_y[train_idx], all_y[val_idx], all_y[test_idx]
gc_train, gc_val, gc_test = all_gc[train_idx], all_gc[val_idx], all_gc[test_idx]

# ── Compute residual targets ─────────────────────────────────────────────────
y_train = y_train - gc_train
y_val = y_val - gc_val
y_test_residual = y_test - gc_test

# ── Fit scalers ───────────────────────────────────────────────────────────────
alt_scaler = MinMaxScaler()
alt_scaler.fit([[0], [12000]])

vel_scaler = StandardScaler()
other_scaler = StandardScaler()
X_train_flat = X_train.reshape(-1, X_train.shape[-1])
vel_scaler.fit(X_train_flat[:, [3, 4]])
other_scaler.fit(X_train_flat[:, [5, 6]])

target_scaler = StandardScaler()
target_scaler.fit(y_train)

print("Target scaler mean:", target_scaler.mean_)
print("Target scaler scale:", target_scaler.scale_)


#save scaler
scaler_folder = "../scalers"
os.makedirs(scaler_folder, exist_ok=True)
joblib.dump(alt_scaler, os.path.join(scaler_folder, "alt_scaler.pkl"))
joblib.dump(vel_scaler, os.path.join(scaler_folder, "vel_scaler.pkl"))
joblib.dump(other_scaler, os.path.join(scaler_folder, "other_scaler.pkl"))
joblib.dump(target_scaler, os.path.join(scaler_folder, "target_scaler.pkl"))



def scale_data(X, is_target=False):
    X = X.copy()
    original_shape = X.shape
    X = X.reshape(-1, X.shape[-1])

    if is_target:
        X = target_scaler.transform(X)
    else:
        X[:, 2:3] = alt_scaler.transform(X[:, 2:3])
        X[:, 3:5] = vel_scaler.transform(X[:, 3:5])
        X[:, 5:7] = other_scaler.transform(X[:, 5:7])

    return X.reshape(original_shape)

def inverse_scale_targets(Y):
    Y = Y.copy()
    original_shape = Y.shape
    Y = Y.reshape(-1, Y.shape[-1])
    Y = target_scaler.inverse_transform(Y)
    return Y.reshape(original_shape)


X_train_scaled = scale_data(X_train, is_target=False)
X_val_scaled   = scale_data(X_val,   is_target=False)
X_test_scaled  = scale_data(X_test,  is_target=False)
y_train_scaled = scale_data(y_train, is_target=True)
y_val_scaled   = scale_data(y_val,   is_target=True)
y_test_scaled  = scale_data(y_test,  is_target=True)

# ── 4. Convert to tensors and create DataLoaders ─────────────────────────────
X_train_t = torch.FloatTensor(X_train_scaled)
y_train_t = torch.FloatTensor(y_train_scaled)
X_val_t   = torch.FloatTensor(X_val_scaled)
y_val_t   = torch.FloatTensor(y_val_scaled)
X_test_t  = torch.FloatTensor(X_test_scaled)
y_test_t  = torch.FloatTensor(y_test_scaled)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=32, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=32, shuffle=False)

Total windows: 866032
X shape: (866032, 10, 7)
y shape: (866032, 3)
gc shape: (866032, 3)
Target scaler mean: [ 7.14672300e-09 -6.74965404e-05 -6.48018780e+00]
Target scaler scale: [3.28932632e-03 7.60705808e-03 4.87829998e+01]


## Create models

In [26]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [27]:
class TrajectoryBiGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=0.3 if num_layers > 1 else 0, bidirectional=True)
        self.attn = nn.Linear(hidden_size * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        out, _ = self.gru(x)                      # (B, 10, H*2)
        weights = torch.softmax(self.attn(out), dim=1)  # (B, 10, 1)
        context = (weights * out).sum(dim=1)       # (B, H*2)
        return self.fc(self.dropout(context))

In [28]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

## Training Multiple config

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Model setup ───────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TrajectoryBiGRU(
    input_size=len(FEATURE_COLS),   
    hidden_size=128,
    num_layers=2,
    output_size=len(TARGET_COLS),  
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.MSELoss()
early_stop = EarlyStopping(patience=25, min_delta=1e-3)

# ── Training loop ─────────────────────────────────────────────────────────────
NUM_EPOCHS = 200

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(X_batch)

    train_loss /= len(train_loader.dataset)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            val_loss += loss.item() * len(X_batch)

    val_loss /= len(val_loader.dataset)
    scheduler.step(val_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if early_stop.step(val_loss):
        print(f"Early stopping at epoch {epoch}")
        break

# ── Testing ───────────────────────────────────────────────────────────────────
model.eval()
test_loss = 0.0
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        test_loss += loss.item() * len(X_batch)

        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

test_loss /= len(test_loader.dataset)

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

# Inverse scale residuals back to real units
preds_residual = inverse_scale_targets(all_preds)
targets_residual = inverse_scale_targets(all_targets)

# Add great-circle guesses to get actual positions
preds_real = preds_residual + gc_test
targets_real = targets_residual + gc_test

# Per-column MAE
for i, col in enumerate(TARGET_COLS):
    mae = np.mean(np.abs(preds_real[:, i] - targets_real[:, i]))
    print(f"{col:15s} MAE: {mae:.4f}")

# Also show how much the GC baseline alone gets you
gc_mae = np.mean(np.abs(gc_test - targets_real), axis=0)
for i, col in enumerate(TARGET_COLS):
    print(f"{col:15s} GC-only MAE: {gc_mae[i]:.4f}")

print(f"\nTest MSE (scaled): {test_loss:.6f}")

Epoch   0 | Train Loss: 0.453333 | Val Loss: 0.356329 | LR: 0.001000
Epoch  10 | Train Loss: 0.334802 | Val Loss: 0.308834 | LR: 0.001000
Epoch  20 | Train Loss: 0.318082 | Val Loss: 0.290706 | LR: 0.001000
Epoch  30 | Train Loss: 0.273243 | Val Loss: 0.282282 | LR: 0.000250
Epoch  40 | Train Loss: 0.249338 | Val Loss: 0.285054 | LR: 0.000125
Epoch  50 | Train Loss: 0.239141 | Val Loss: 0.287082 | LR: 0.000031
Early stopping at epoch 55
latitude        MAE: 0.0120
longitude       MAE: 0.0434
altitude        MAE: 5.1574
latitude        GC-only MAE: 0.0119
longitude       GC-only MAE: 0.0433
altitude        GC-only MAE: 20.1632

Test MSE (scaled): 18.782296


In [30]:
torch.save(model.state_dict(), "../models/BiGRU.pth")

In [45]:
def predict_gap(model, before_df, after_df, n_steps, feature_cols, target_cols,
                sequence_len, device, save_path=None):
    model.eval()

    def haversine_dist(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return 2 * 6371000 * np.arcsin(np.sqrt(a))

    def bearing(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlon = lon2 - lon1
        x = np.sin(dlon) * np.cos(lat2)
        y = np.cos(lat1)*np.sin(lat2) - np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
        return np.degrees(np.arctan2(x, y)) % 360

    def great_circle_extrapolate(lat1, lon1, lat2, lon2):
        lat1r, lon1r, lat2r, lon2r = map(np.radians, [lat1, lon1, lat2, lon2])
        dlon = lon2r - lon1r
        x = np.cos(lat2r) * np.sin(dlon)
        y = np.cos(lat1r)*np.sin(lat2r) - np.sin(lat1r)*np.cos(lat2r)*np.cos(dlon)
        brng = np.arctan2(x, y)
        dlat = lat2r - lat1r
        a = np.sin(dlat/2)**2 + np.cos(lat1r)*np.cos(lat2r)*np.sin(dlon/2)**2
        ang_dist = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        lat3 = np.arcsin(np.sin(lat2r)*np.cos(ang_dist) +
                         np.cos(lat2r)*np.sin(ang_dist)*np.cos(brng))
        lon3 = lon2r + np.arctan2(np.sin(brng)*np.sin(ang_dist)*np.cos(lat2r),
                                   np.cos(ang_dist) - np.sin(lat2r)*np.sin(lat3))
        return np.degrees(lat3), np.degrees(lon3)

    def recompute_vx_vy_acc_tr(context, dt=15.0):
        n = len(context)
        if n < 2:
            return context
        lat1, lon1 = context[-2, 0], context[-2, 1]
        lat2, lon2 = context[-1, 0], context[-1, 1]
        heading = bearing(lat1, lon1, lat2, lon2)
        velocity = haversine_dist(lat1, lon1, lat2, lon2) / dt
        heading_rad = np.deg2rad(heading)
        context[-1, 3] = velocity * np.sin(heading_rad)
        context[-1, 4] = velocity * np.cos(heading_rad)
        if n >= 3:
            prev_lat, prev_lon = context[-3, 0], context[-3, 1]
            prev_heading = bearing(prev_lat, prev_lon, lat1, lon1)
            prev_velocity = haversine_dist(prev_lat, prev_lon, lat1, lon1) / dt
            context[-1, 5] = (velocity - prev_velocity) / dt
            context[-1, 6] = (heading - prev_heading) / dt
        else:
            context[-1, 5] = 0.0
            context[-1, 6] = 0.0
        return context

    def prepare_context(df, feature_cols, sequence_len, from_end=True):
        if from_end:
            ctx = df[feature_cols].values[-sequence_len:].copy()
        else:
            ctx = df[feature_cols].values[:sequence_len].copy()
        for col in range(ctx.shape[1]):
            mask = np.isnan(ctx[:, col])
            if mask.all():
                ctx[:, col] = 0.0
                continue
            for i in range(1, len(ctx)):
                if np.isnan(ctx[i, col]):
                    ctx[i, col] = ctx[i-1, col]
            for i in range(len(ctx)-2, -1, -1):
                if np.isnan(ctx[i, col]):
                    ctx[i, col] = ctx[i+1, col]
        return ctx

    def predict_direction(context, n_steps, reverse=False):
        preds = []
        ctx = context.copy()
        first_step = True  # add this
        
        for _ in range(n_steps):
            ref_lat = ctx[-1, 0]
            ref_lon = ctx[-1, 1]
            ref_alt = ctx[-1, 2]

            # Great-circle extrapolation from last two points
            gc_lat, gc_lon = great_circle_extrapolate(
                ctx[-2, 0], ctx[-2, 1], ctx[-1, 0], ctx[-1, 1]
            )
            gc_alt = ref_alt

            # GC guess relative to reference
            gc_rel = np.array([gc_lat - ref_lat, gc_lon - ref_lon, gc_alt - ref_alt])

            # Make window relative
            window = ctx.copy()
            window[:, 0] -= ref_lat
            window[:, 1] -= ref_lon
            window[:, 2] -= ref_alt

            window_scaled = scale_data(window.reshape(1, sequence_len, -1), is_target=False)
            input_tensor = torch.FloatTensor(window_scaled).to(device)

            with torch.no_grad():
                pred_scaled = model(input_tensor).cpu().numpy()

            # Model predicts residual from GC guess
            pred_residual = inverse_scale_targets(pred_scaled)[0]

            # Final position = GC guess + residual (in relative space), then add ref
            pred_abs = np.array([
                gc_rel[0] + pred_residual[0] + ref_lat,
                gc_rel[1] + pred_residual[1] + ref_lon,
                gc_rel[2] + pred_residual[2] + ref_alt
            ])

            
            preds.append(pred_abs)

            new_row = np.zeros(len(feature_cols))
            new_row[0] = pred_abs[0]
            new_row[1] = pred_abs[1]
            new_row[2] = pred_abs[2]

            ctx = np.vstack([ctx[1:], new_row])
            ctx = recompute_vx_vy_acc_tr(ctx)

        return np.array(preds)

    # ── Forward prediction ───────────────────────────────────────────────────
    context_fwd = prepare_context(before_df, feature_cols, sequence_len, from_end=True)
    forward_preds = predict_direction(context_fwd, n_steps)

    # ── Backward prediction ──────────────────────────────────────────────────
    context_bwd = prepare_context(after_df, feature_cols, sequence_len, from_end=False)
    context_bwd = context_bwd[::-1].copy()
    backward_preds = predict_direction(context_bwd, n_steps)
    backward_preds = backward_preds[::-1]

    # ── Blend ────────────────────────────────────────────────────────────────
    alpha = np.linspace(1.0, 0.0, n_steps).reshape(-1, 1)
    blended = alpha * forward_preds + (1 - alpha) * backward_preds

    # ── Build prediction output ──────────────────────────────────────────────
    last_time = before_df["timestamp"].iloc[-1]
    timestamps = [last_time + pd.Timedelta(seconds=15 * (i + 1)) for i in range(n_steps)]

    result = pd.DataFrame(blended, columns=target_cols)
    result["timestamp"] = timestamps
    result["interpolated"] = True

    # ── Save full reconstructed trajectory ───────────────────────────────────
    if save_path is not None:
        alt_col = "altitude" if "altitude" in before_df.columns else "geoaltitude_m"

        before_out = before_df[["timestamp", "latitude", "longitude"]].copy()
        before_out["altitude"] = before_df[alt_col] if alt_col in before_df.columns else np.nan
        before_out["source"] = "adsb_before"

        after_out = after_df[["timestamp", "latitude", "longitude"]].copy()
        after_out["altitude"] = after_df[alt_col] if alt_col in after_df.columns else np.nan
        after_out["source"] = "adsb_after"

        pred_out = result[["timestamp", "latitude", "longitude", "altitude"]].copy()
        pred_out["source"] = "predicted"

        full_trajectory = pd.concat([before_out, pred_out, after_out], ignore_index=True)
        full_trajectory = full_trajectory.sort_values("timestamp").reset_index(drop=True)
        full_trajectory.to_parquet(save_path)
        print(f"Saved reconstructed trajectory ({len(full_trajectory)} points) to {save_path}")

    return result

In [46]:
# Remove point from a flight and see how well the model reconstructs it

flight = df_list[0].copy()
flight = flight.sort_values("timestamp").reset_index(drop=True)

cut_start = 25
cut_end = 175
before = flight.iloc[:cut_start]
hidden = flight.iloc[cut_start:cut_end]
after  = flight.iloc[cut_end:]

preds = predict_gap(
    model, before, after,
    n_steps=cut_end - cut_start,
    feature_cols=FEATURE_COLS,
    target_cols=TARGET_COLS,
    sequence_len=SEQUENCE_LEN,
    device=device,
    save_path="reconstructed_trajectory.parquet"
)

# Per-step error
pred_lats = preds["latitude"].values
true_lats = hidden["latitude"].values
pred_lons = preds["longitude"].values
true_lons = hidden["longitude"].values
pred_lats = preds["altitude"].values

for step in range(len(hidden)):
    if step%5==0 or step==len(hidden)-1:
        print(f"Step {step:2d} | Lat err: {abs(pred_lats[step] - true_lats[step]):.4f}° | Lon err: {abs(pred_lons[step] - true_lons[step]):.4f}|°")

# Compare
print("Lat MAE:", np.mean(np.abs(preds["latitude"].values - hidden["latitude"].values)))
print("Lon MAE:", np.mean(np.abs(preds["longitude"].values - hidden["longitude"].values)))


Saved reconstructed trajectory (199 points) to reconstructed_trajectory.parquet
Step  0 | Lat err: 10319.2867° | Lon err: 0.0509|°
Step  5 | Lat err: 10254.4784° | Lon err: 0.1768|°
Step 10 | Lat err: 10208.7183° | Lon err: 0.3771|°
Step 15 | Lat err: 10177.4258° | Lon err: 0.4556|°
Step 20 | Lat err: 10170.6061° | Lon err: 0.3463|°
Step 25 | Lat err: 10178.9586° | Lon err: 0.3922|°
Step 30 | Lat err: 10205.5421° | Lon err: 0.3350|°
Step 35 | Lat err: 10246.2918° | Lon err: 0.3337|°
Step 40 | Lat err: 10299.6428° | Lon err: 0.3379|°
Step 45 | Lat err: 10301.9131° | Lon err: 0.3328|°
Step 50 | Lat err: 10302.7024° | Lon err: 0.3764|°
Step 55 | Lat err: 10303.2147° | Lon err: 0.5560|°
Step 60 | Lat err: 10303.6337° | Lon err: 0.7310|°
Step 65 | Lat err: 10303.7662° | Lon err: 0.9022|°
Step 70 | Lat err: 10303.6291° | Lon err: 1.0086|°
Step 75 | Lat err: 10303.6111° | Lon err: 0.8667|°
Step 80 | Lat err: 10303.7403° | Lon err: 0.7225|°
Step 85 | Lat err: 10304.0155° | Lon err: 0.5765|°
St

In [ ]:
file_list[0]

,timestamp,icao24,latitude,longitude,velocity_mps,heading_deg,geoaltitude_m,baroaltitude_m,callsign,lastcontact,onground,altitude,delta_t,vx,vy,acceleration,turn_rate
0,2023-08-10 10:46:15,4ba959,46.926238,-62.602351,NaN,NaN,10584.18,10363.2,<NA>,1.691654e+09,0.0,10363.2,NaN,NaN,NaN,NaN,NaN
1,2023-08-10 10:46:30,4ba959,46.922836,-62.610672,49.201025,239.173740,10584.18,10363.2,<NA>,1.691654e+09,0.0,10363.2,15.0,-42.250156,-25.212400,NaN,NaN
2,2023-08-10 10:46:45,4ba959,46.911295,-62.638902,166.936274,239.188078,10584.18,10363.2,<NA>,1.691654e+09,0.0,10363.2,15.0,-143.373775,-85.508363,7.849017,0.000956
3,2023-08-10 10:47:00,4ba959,46.903055,-62.659044,119.142855,239.173352,10584.18,10363.2,<NA>,1.691654e+09,0.0,10363.2,15.0,-102.310549,-61.053840,-3.186228,-0.000982
4,2023-08-10 10:47:15,4ba959,46.870609,-62.738061,467.999481,239.109604,10584.18,10363.2,<NA>,1.691654e+09,0.0,10363.2,15.0,-401.614209,-240.269727,23.257108,-0.004250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,2023-08-10 11:34:45,4ba959,43.486176,-69.843282,225.661021,234.642207,10660.38,10363.2,THY181,1.691656e+09,0.0,10363.2,15.0,-184.038817,-130.585642,-0.008331,-0.007450
195,2023-08-10 11:35:00,4ba959,43.468385,-69.877625,227.326567,234.585126,10660.38,10363.2,THY181,1.691656e+09,0.0,10363.2,15.0,-185.266012,-131.734099,0.111036,-0.003805
196,2023-08-10 11:35:15,4ba959,43.450470,-69.912266,229.214337,234.638953,10668.00,10363.2,THY181,1.691657e+09,0.0,10363.2,15.0,-186.929207,-132.652494,0.125851,0.003588
197,2023-08-10 11:35:30,4ba959,43.432755,-69.946374,226.061849,234.530633,10660.38,10363.2,THY181,1.691657e+09,0.0,10363.2,15.0,-184.110620,-131.176368,-0.210166,-0.007221
